In [1]:
import faiss
import pandas as pd
import numpy as np

from pathlib import Path
import re
import html
import time
import requests
import numpy as np
import pandas as pd
import json

PROJECT_ROOT = Path.cwd()
CORPUS_ROOT = PROJECT_ROOT / "semiconductor_patents"

CATEGORIES = [
    "semiconductor_manufacturing",
    "transistor_device_technology",
    "memory",
    "advanced_packaging",
    "power_semiconductors",
    "photonics",
    "ai_advanced_semiconductor",
]

EMBED_MODEL = "nomic-embed-text"
LLM_MODEL = "qwen3:8b"
OLLAMA_BASE_URL = "http://localhost:11434"

CHUNK_SIZE = 1500
CHUNK_OVERLAP = 250
TOP_K = 5
CANDIDATE_K = 30

print("Project root:", PROJECT_ROOT)
print("Corpus root :", CORPUS_ROOT)
print("Embedding   :", EMBED_MODEL)
print("LLM         :", LLM_MODEL)


OLLAMA_EMBED_URL = f"{OLLAMA_BASE_URL}/api/embeddings"
OLLAMA_GENERATE_URL = f"{OLLAMA_BASE_URL}/api/generate"

def ollama_json(url, payload, timeout=120):
    response = requests.post(url, json=payload, timeout=timeout)
    response.raise_for_status()
    return response.json()

def embed_text(text):
    result = ollama_json(
        OLLAMA_EMBED_URL,
        {"model": EMBED_MODEL, "prompt": text},
    )
    return np.asarray(result["embedding"], dtype=np.float32)

def ask_ollama(prompt):
    result = ollama_json(
        OLLAMA_GENERATE_URL,
        {
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
        },
        timeout=180,
    )
    return result["response"]

tags = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=10)
tags.raise_for_status()
available_models = {m["name"] for m in tags.json().get("models", [])}

print("Ollama connection: OK")
print("Embedding model available:", EMBED_MODEL in available_models)
print("LLM model available:", LLM_MODEL in available_models)

Project root: /Users/jamesjr/Documents/SemiconductorPatentRAG
Corpus root : /Users/jamesjr/Documents/SemiconductorPatentRAG/semiconductor_patents
Embedding   : nomic-embed-text
LLM         : qwen3:8b
Ollama connection: OK
Embedding model available: False
LLM model available: True


In [20]:
# -------------------------------------------------
# Load once (at the top of your notebook / script)
# -------------------------------------------------
index = faiss.read_index("embeddings.faiss")
meta  = pd.read_parquet("chunks_meta.parquet")   # contains ALL original columns




# meta already loaded:
# meta = pd.read_parquet("chunks_meta.parquet")

def rebuild_patents_df(meta: pd.DataFrame) -> pd.DataFrame:
    """
    Reconstruct a document-level patents_df from the chunk metadata,
    preserving all original columns.
    """
    # Group by document and aggregate
    patents = (
        meta
        .groupby("document_id", as_index=False)
        .agg({
            "category": "first",
            "cpc_section": "first",
            "source": "first",
            "file_path": "first",
            "text": lambda texts: "\n\n".join(texts),   # full reconstructed text
            "chunk_id": "count",
        })
        .rename(columns={
            "text": "full_text",
            "chunk_id": "num_chunks"
        })
    )

    # Add the missing original columns (filled as well as possible)
    patents["filename"] = patents["file_path"].apply(lambda x: Path(x).name if pd.notna(x) else None)
    
    # These three were section-level in the original parser.
    # We cannot perfectly recover them from chunks, so we leave them empty
    # (or you can later fill them from the original files if still available).
    patents["abstract"] = ""
    patents["description"] = ""
    patents["claims"] = ""

    # Approximate word / character counts from the reconstructed text
    patents["word_count"] = patents["full_text"].str.count(r"\b\w+\b")
    patents["character_count"] = patents["full_text"].str.len()

    # Re-order columns to match your original patents_df as closely as possible
    desired_order = [
        "document_id",
        "category",
        "cpc_section",
        "source",
        "abstract",
        "description",
        "claims",
        "filename",
        "file_path",
        "word_count",
        "character_count",
        "num_chunks",      # extra useful column
        "full_text",       # extra useful column
    ]
    
    # Keep any extra columns that may exist
    extra_cols = [c for c in patents.columns if c not in desired_order]
    patents = patents[desired_order + extra_cols]

    return patents


# Usage
patents_df = rebuild_patents_df(meta)
print(patents_df.shape)
patents_df.head()

(500, 13)


,document_id,category,cpc_section,source,abstract,description,claims,filename,file_path,word_count,character_count,num_chunks,full_text
0,000a442355b67fd1,transistor_device_technology,,BIGPATENT,,,,patent_000a442355b67fd1.txt,/Users/jamesjr/Documents/SemiconductorPatentRA...,7609,43979,30,To provide a magnetic sensor device which main...
1,0091d26ac8fe4c88,transistor_device_technology,,BIGPATENT,,,,patent_0091d26ac8fe4c88.txt,/Users/jamesjr/Documents/SemiconductorPatentRA...,7013,41346,28,A liquid crystal display device with a display...
2,01ea3f6e9244856597cb,advanced_packaging,H,BIGPATENT,,,,patent_01ea3f6e9244856597cb.txt,/Users/jamesjr/Documents/SemiconductorPatentRA...,3476,21767,18,Embodiments of the invention include a semicon...
3,027109fedf9f0c67,memory,,BIGPATENT,,,,patent_027109fedf9f0c67.txt,/Users/jamesjr/Documents/SemiconductorPatentRA...,4029,23043,16,An electrical circuit and method to compare co...
4,034501f4ef2b8771,transistor_device_technology,,BIGPATENT,,,,patent_034501f4ef2b8771.txt,/Users/jamesjr/Documents/SemiconductorPatentRA...,8420,49648,34,A switching circuit has a first Field Effect T...


## 6. Semantic retrieval with patent diversity

The first retrieval stage ranks chunks by cosine similarity. We then remove duplicate patents so one patent cannot dominate the answer.

In [21]:
# -------------------------------------------------
# Updated retrieval function
# -------------------------------------------------
def retrieve_patents(query, top_k=TOP_K, candidate_k=CANDIDATE_K):
    # Embed + normalize query
    query_vector = np.asarray(embed_text(query), dtype=np.float32)
    query_vector = query_vector / max(np.linalg.norm(query_vector), 1e-12)
    query_vector = query_vector.reshape(1, -1)

    # FAISS search (Inner Product == cosine because vectors are normalized)
    scores, indices = index.search(query_vector, candidate_k)
    scores = scores[0]
    indices = indices[0]

    selected = []
    seen = set()

    for score, idx in zip(scores, indices):
        if idx < 0:                       # FAISS returns -1 when not enough results
            continue

        row = meta.iloc[idx]
        patent_id = row["document_id"]

        if patent_id in seen:
            continue
        seen.add(patent_id)

        selected.append({
            "document_id": patent_id,
            "category": row["category"],
            "cpc_section": row["cpc_section"],
            "source": row["source"],
            "section": row["section"],
            "chunk_id": row["chunk_id"],
            "text": row["text"],
            "similarity": float(score),
            "file_path": row["file_path"],
        })

        if len(selected) == top_k:
            break

    return pd.DataFrame(selected)


def show_results(results, preview=500):
    display(
        results[["document_id", "category", "section", "similarity"]]
    )
    for rank, (_, row) in enumerate(results.iterrows(), 1):
        print(f"{rank}. {row['document_id']} | {row['category']} | {row['similarity']:.4f}")
        print(row["text"][:preview].replace("\n", " "))
        print()

In [22]:
query = "What methods improve semiconductor manufacturing yield and defect detection?"
results = retrieve_patents(query)
show_results(results)

,document_id,category,section,similarity
0,d3ead82de489632b,semiconductor_manufacturing,DESCRIPTION,0.807123
1,678e9b7f28681857,semiconductor_manufacturing,DESCRIPTION,0.797923
2,1bc765dc24dc5424,semiconductor_manufacturing,DESCRIPTION,0.760105
3,eccc29daa65f9d0d,transistor_device_technology,DESCRIPTION,0.750600
4,f25e95d6e030dc83,transistor_device_technology,DESCRIPTION,0.729112


1. d3ead82de489632b | semiconductor_manufacturing | 0.8071
o the theoretical number of devices that could be produced assuming no bad devices. Improving yield is a critical problem in the semiconductor manufacturing industry and has a direct economic impact on it. In particular, a higher yield translates into more devices that may be sold by the manufacturer, and greater profits. Typically, semiconductor manufacturers collect data about various defects and analyze the data and, based on data analysis, adjust the integrated circuit design or process step

2. 678e9b7f28681857 | semiconductor_manufacturing | 0.7979
BACKGROUND OF THE INVENTION 1. Field of the Invention This invention relates generally to the manufacturing of high performance semiconductor devices. More specifically, this invention relates to a method of isolating optical defect images that have been captured during inspection for defects. 2. Discussion of the Related Art In order to remain competitive, a semiconductor manu

## 7. Evidence-aware RAG

The answer generator receives only retrieved evidence and must cite the document IDs supplied to it. Claims are not fabricated when the corpus does not contain them.

In [23]:
def build_evidence_context(results):
    blocks = []

    for number, (_, row) in enumerate(results.iterrows(), 1):
        patent = patents_df[
            patents_df.document_id == row["document_id"]
        ].iloc[0]

        abstract = patent["abstract"] or "No abstract available."
        claims = patent["claims"] or "No claims available."

        block = "\n".join([
            f"SOURCE {number}",
            "=====================",
            f"PATENT ID: {row['document_id']}",
            f"CATEGORY: {row['category']}",
            f"CPC SECTION: {row['cpc_section'] or 'Not available'}",
            f"SIMILARITY: {row['similarity']:.4f}",
            "",
            "ABSTRACT:",
            abstract[:2000],
            "",
            "CLAIMS:",
            claims[:3000],
            "",
            "RETRIEVED PATENT TEXT:",
            row["text"][:3500],
            "",
            "END SOURCE",
            "=====================",
        ])
        blocks.append(block)

    return "\n\n".join(blocks)

def build_rag_prompt(question, context):
    return (
        "You are a semiconductor patent research assistant.\n\n"
        "Answer using ONLY the retrieved patent evidence below.\n\n"
        "Rules:\n"
        "1. Do not use outside knowledge.\n"
        "2. Do not invent patent IDs, claims, technical details, or conclusions.\n"
        "3. Support technical statements with the supplied evidence.\n"
        "4. Cite supporting patents as [Patent: DOCUMENT_ID].\n"
        "5. If evidence is insufficient, say so explicitly.\n"
        "6. Do not make legal conclusions about validity, infringement, novelty, or ownership.\n"
        "7. Do not claim that a patent has claims when the supplied corpus has none.\n\n"
        "RETRIEVED PATENT EVIDENCE\n"
        "=========================\n"
        f"{context}\n\n"
        "QUESTION\n"
        "========\n"
        f"{question}\n\n"
        "ANSWER\n"
        "======"
    )

## 8. End-to-end RAG query

**Question → embedding → retrieval → evidence context → grounded prompt → local Qwen model**

In [24]:
def run_rag(question, top_k=TOP_K, candidate_k=CANDIDATE_K):
    results = retrieve_patents(
        question,
        top_k=top_k,
        candidate_k=candidate_k,
    )

    if results.empty:
        return {
            "question": question,
            "answer": "No relevant patent evidence was retrieved.",
            "results": results,
            "context": "",
        }

    context = build_evidence_context(results)
    prompt = build_rag_prompt(question, context)
    answer = ask_ollama(prompt)

    return {
        "question": question,
        "answer": answer,
        "results": results,
        "context": context,
    }

question = (
    "What techniques are used to improve electrical performance "
    "and signal integrity in advanced semiconductor packaging?"
)

rag_result = run_rag(question)

print("=" * 72)
print("RAG ANSWER")
print("=" * 72)
print(rag_result["answer"])

print("\nPATENTS USED")
for rank, (_, row) in enumerate(rag_result["results"].iterrows(), 1):
    print(
        f"{rank}. {row['document_id']} | "
        f"{row['category']} | "
        f"similarity={row['similarity']:.4f}"
    )

RAG ANSWER
The techniques described in the retrieved patents for improving electrical performance and signal integrity in advanced semiconductor packaging include:  

1. **Optimized Wiring Configurations on Interposer Substrates**: The patent highlights wiring configurations on an interposer substrate designed for high-speed signal transmission, which likely involves reducing signal interference and impedance mismatches to maintain integrity [Patent: a969f5fbe709d62ce477].  

2. **Fan-Out Packaging Technologies**: This approach addresses miniaturization and reliability by redistributing interconnects, which can mitigate signal degradation through improved routing and reduced crosstalk [Patent: 5c06202f88b554e8144b].  

3. **Multi-Chip Package Integration**: Techniques such as integrating multiple chips (e.g., memory or drawing chips) into a single package with high operating frequency and low cost are mentioned, implying methods to maintain signal integrity across heterogeneous compone

## 9. Retrieval benchmark

This retains the useful category-hit evaluation from the original notebook, while clearly treating it as a retrieval sanity check rather than a full RAG accuracy metric.

In [25]:
evaluation_set = {
    "What techniques are used to reduce impedance in semiconductor packaging?":
        {"advanced_packaging"},
    "What methods are used for semiconductor wafer defect detection?":
        {"semiconductor_manufacturing"},
    "How are thermal management problems addressed in semiconductor packages?":
        {"advanced_packaging"},
    "What techniques are used to improve semiconductor interconnect reliability?":
        {"advanced_packaging", "semiconductor_manufacturing"},
    "What methods are used to improve semiconductor manufacturing yield?":
        {"semiconductor_manufacturing"},
    "How are semiconductor devices packaged to reduce signal loss?":
        {"advanced_packaging"},
    "What techniques are used for wafer-level packaging?":
        {"advanced_packaging"},
    "How are semiconductor manufacturing defects identified?":
        {"semiconductor_manufacturing"},
    "What methods are used to improve heat dissipation in semiconductor devices?":
        {"advanced_packaging"},
    "What approaches are used to improve electrical performance in advanced semiconductor packaging?":
        {"advanced_packaging"},
}

rows = []

for question, expected in evaluation_set.items():
    retrieved = retrieve_patents(question)
    categories = set(retrieved.category)

    rows.append({
        "question": question,
        "expected": ", ".join(sorted(expected)),
        "top1_category": retrieved.iloc[0]["category"] if not retrieved.empty else None,
        "retrieved_categories": ", ".join(sorted(categories)),
        "category_hit": bool(expected & categories),
    })

evaluation_df = pd.DataFrame(rows)
display(evaluation_df)

hit_rate = evaluation_df.category_hit.mean()
print(f"Category hit rate: {hit_rate * 100:.1f}%")

,question,expected,top1_category,retrieved_categories,category_hit
0,What techniques are used to reduce impedance i...,advanced_packaging,advanced_packaging,"advanced_packaging, power_semiconductors",True
1,What methods are used for semiconductor wafer ...,semiconductor_manufacturing,semiconductor_manufacturing,"advanced_packaging, semiconductor_manufacturin...",True
2,How are thermal management problems addressed ...,advanced_packaging,advanced_packaging,"advanced_packaging, semiconductor_manufacturing",True
3,What techniques are used to improve semiconduc...,"advanced_packaging, semiconductor_manufacturing",advanced_packaging,advanced_packaging,True
4,What methods are used to improve semiconductor...,semiconductor_manufacturing,semiconductor_manufacturing,"advanced_packaging, semiconductor_manufacturing",True
5,How are semiconductor devices packaged to redu...,advanced_packaging,advanced_packaging,advanced_packaging,True
6,What techniques are used for wafer-level packa...,advanced_packaging,advanced_packaging,"advanced_packaging, transistor_device_technology",True
7,How are semiconductor manufacturing defects id...,semiconductor_manufacturing,semiconductor_manufacturing,"semiconductor_manufacturing, transistor_device...",True
8,What methods are used to improve heat dissipat...,advanced_packaging,semiconductor_manufacturing,"advanced_packaging, power_semiconductors, semi...",True
9,What approaches are used to improve electrical...,advanced_packaging,advanced_packaging,"advanced_packaging, power_semiconductors",True


Category hit rate: 100.0%


## 10. Final baseline summary

The original notebooks contained duplicated parsers, multiple retrieval functions, multiple prompt builders, stale metadata experiments, two embedding approaches, and a broken `rerank_k` call.

This clean baseline keeps the working conceptual components while removing that experimental duplication. Reranking can be added later as a separate measured stage rather than being silently mixed into the baseline.

In [32]:
summary = {
    "documents": int(patents_df.document_id.nunique()),
    "chunks": int(len(meta)),
    "categories": int(patents_df.category.nunique()),
    "embedding_model": EMBED_MODEL,
    "embedding_dimensions": int(index.d),          # ← fixed (FAISS uses .d, not .shape)
    "llm_model": LLM_MODEL,
    "claims_available": int(patents_df.claims.ne("").sum()),
}

In [33]:
summary

{'documents': 500,
 'chunks': 13220,
 'categories': 7,
 'embedding_model': 'nomic-embed-text',
 'embedding_dimensions': 768,
 'llm_model': 'qwen3:8b',
 'claims_available': 0}